# Housing Price Classification

Goal: predict whether a house is classified as expensive using housing features.

Target variable: Expensive

Problem type: binary classification

Final output: predictions for the competition test set

## train/test split, imputing missing values and preprocessing pipeline ##

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import make_column_transformer
from sklearn import set_config
set_config(transform_output='pandas')

data = pd.read_csv("data/housing_iteration_5_classification.csv")

In [130]:
data.columns

Index(['LotArea', 'LotFrontage', 'TotalBsmtSF', 'BedroomAbvGr', 'Fireplaces',
       'PoolArea', 'GarageCars', 'WoodDeckSF', 'ScreenPorch', 'Expensive',
       'MSZoning', 'Condition1', 'Heating', 'Street', 'CentralAir',
       'Foundation', 'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
       'BsmtExposure', 'BsmtFinType1', 'KitchenQual', 'FireplaceQu',
       'MSSubClass', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', '1stFlrSF',
       '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath',
       'FullBath', 'HalfBath', 'KitchenAbvGr', 'TotRmsAbvGrd', 'GarageYrBlt',
       'GarageArea', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'MiscVal',
       'MoSold', 'YrSold', 'Id', 'Alley', 'LotShape', 'LandContour',
       'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrTyp

In [131]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   LotArea        1460 non-null   int64  
 1   LotFrontage    1201 non-null   float64
 2   TotalBsmtSF    1460 non-null   int64  
 3   BedroomAbvGr   1460 non-null   int64  
 4   Fireplaces     1460 non-null   int64  
 5   PoolArea       1460 non-null   int64  
 6   GarageCars     1460 non-null   int64  
 7   WoodDeckSF     1460 non-null   int64  
 8   ScreenPorch    1460 non-null   int64  
 9   Expensive      1460 non-null   int64  
 10  MSZoning       1460 non-null   str    
 11  Condition1     1460 non-null   str    
 12  Heating        1460 non-null   str    
 13  Street         1460 non-null   str    
 14  CentralAir     1460 non-null   str    
 15  Foundation     1460 non-null   str    
 16  ExterQual      1460 non-null   str    
 17  ExterCond      1460 non-null   str    
 18  BsmtQual       1423

In [ ]:
# Drop columns that are not useful for prediction

data.drop(columns=["Alley", "PoolQC", "MiscFeature", "Id"], inplace=True)

# Dropped Id because it is only an identifier.
# Dropped Alley, PoolQC, and MiscFeature because they contain many missing values.

In [133]:
# Separate the target variable from the features

X = data.copy()
y = X.pop("Expensive")


# Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=42)

In [134]:
# Preprocessing Pipelines

# Identify numerical and categorical features
num_features = X.select_dtypes(include="number").columns
cat_features = X.select_dtypes(exclude="number").columns

# Numerical pipeline: Impute missing values using the mean.
num_pipe = make_pipeline(SimpleImputer(strategy="mean"))

# Categorical preprocessing:
# - Impute missing categorical values with a constant 'N_A'
cat_imputer = SimpleImputer(strategy="constant", fill_value="N_A")

In [135]:
# For ordinal encoding, specify the feature and its explicit order.
ordinal_features = ["Utilities", "ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "BsmtFinType1", "BsmtFinType2", "HeatingQC", "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "Fence", "BsmtExposure", "GarageFinish", "PavedDrive", "LotShape", "LandSlope", "Functional"]
utilities_categories = ["N_A", "ELO", "NoSeWa", "NoSewr", "AllPub"]
exqual_categories = ["N_A", "Po", "Fa", "TA", "Gd", "Ex"]
excond_categories = ["N_A", "Po", "Fa", "TA", "Gd", "Ex"]
bsmtqual_categories = ["N_A", "Po", "Fa", "TA", "Gd", "Ex"]
bsmtcond_categories = ["N_A", "Po", "Fa", "TA", "Gd", "Ex"]
bsmtfintype1_categories = ["N_A", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"]
bsmtfintype2_categories = ["N_A", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"]
heatingqc_categories = ["N_A", "Po", "Fa", "TA", "Gd", "Ex"]
kitchenqual_categories = ["N_A", "Po", "Fa", "TA", "Gd", "Ex"]
fireplacequ_categories = ["N_A", "Po", "Fa", "TA", "Gd", "Ex"]
garagequal_categories = ["N_A", "Po", "Fa", "TA", "Gd", "Ex"]
garagecond_categories = ["N_A", "Po", "Fa", "TA", "Gd", "Ex"]
fence_categories = ["N_A", "MnWw", "GdWo", "MnPrv", "GdPrv"]
bsmtexposure_categories = ["N_A", "No", "Mn", "Av", "Gd"]
garagefinish_categories = ["N_A", "Unf", "RFn", "Fin"]
paveddrive_categories = ["N_A", "N", "P", "Y"]
lotshape_categories = ["N_A", "IR3", "IR2", "IR1", "Reg"]
landslope_categories = ["N_A", "Sev", "Mod", "Gtl"]
functional_categories = ["N_A", "Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"]

ord_encoder = OrdinalEncoder(categories=[utilities_categories, exqual_categories, excond_categories, bsmtqual_categories, bsmtcond_categories, bsmtfintype1_categories, bsmtfintype2_categories, heatingqc_categories, kitchenqual_categories, fireplacequ_categories, garagequal_categories, garagecond_categories, fence_categories, bsmtexposure_categories, garagefinish_categories, paveddrive_categories, lotshape_categories, landslope_categories, functional_categories])


In [136]:
# Identify nominal categorical features (those that will be one-hot encoded)
nominal_features = list(set(cat_features) - set(ordinal_features))
oh_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# Combine ordinal and nominal encoders using a column transformer.
cat_encoder = make_column_transformer(
    (ord_encoder, ordinal_features),
    (oh_encoder, nominal_features)
)

# Create a categorical pipeline: imputation followed by encoding.
cat_pipe = make_pipeline(cat_imputer, cat_encoder)

# Combine numerical and categorical pipelines into a full preprocessor.
preprocessor = make_column_transformer(
    (num_pipe, num_features),
    (cat_pipe, cat_features)
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('pipeline-1', ...), ('pipeline-2', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_

## Decision Tree ##

In [137]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier

# Define a function to perform a grid search, which helps to avoid duplicating code for different models
def run_grid_search(model, param_grid, X_train, y_train, preprocessor, scaler=None, cv=5, verbose=1):
    # Create a pipeline that first applies the data preprocessing steps, then fits the model
    if scaler is None:
        pipe = make_pipeline(preprocessor, model)
    else:
        pipe = make_pipeline(preprocessor, scaler, model)

    # GridSearchCV will test all possible combinations of parameters defined in 'param_grid'
    grid_search = GridSearchCV(pipe, param_grid, cv=cv, verbose=verbose)

    # Fit the model on the training data with the various parameter combinations
    grid_search.fit(X_train, y_train)

    # Return the trained GridSearchCV object which holds the best parameters and model
    return grid_search

# Define a dictionary of hyperparameters to tune for the decision tree model
dt_param_grid = {
    "columntransformer__pipeline-1__simpleimputer__strategy": ["mean", "median"],
    "decisiontreeclassifier__max_depth": range(2, 14, 2),
    "decisiontreeclassifier__min_samples_leaf": range(3, 12, 2)
}

# Run the grid search for the DecisionTreeClassifier using the specified parameters
dt_search = run_grid_search(
    DecisionTreeClassifier(random_state=42),
    dt_param_grid,
    X_train,
    y_train,
    preprocessor
)

# Display the process
dt_search

Fitting 5 folds for each of 60 candidates, totalling 300 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'columntransformer__pip...simpleimputer__strategy': ['mean', 'median'], 'decisiontreeclassifier__max_depth': range(2, 14, 2), 'decisiontreeclassifier__min_samples_leaf': range(3, 12, 2)}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the mor

In [138]:
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    balanced_accuracy_score,
    cohen_kappa_score
)

# Function to get the scores for our model(s)
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    scores = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Specificity": recall_score(y_test, y_pred, pos_label=0),
        "F1 Score": f1_score(y_test, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "Cohen's Kappa": cohen_kappa_score(y_test, y_pred)
    }
    return scores

# Create an empty DataFrame to store model evaluation results
model_scores_df = pd.DataFrame(columns=[
    "Model", "Accuracy", "Recall", "Precision",
    "Specificity", "F1 Score", "Balanced Accuracy", "Cohen's Kappa"
])

# Evaluate the Decision Tree model
dt_scores = evaluate_model(dt_search, X_test, y_test)
dt_scores["Model"] = "Decision Tree"

# Convert the dictionary to a Series matching the DataFrame columns, then assign as a new row
model_scores_df.loc[len(model_scores_df)] = pd.Series(dt_scores, index=model_scores_df.columns)

# Display the DataFrame
model_scores_df

,Model,Accuracy,Recall,Precision,Specificity,F1 Score,Balanced Accuracy,Cohen's Kappa
0,Decision Tree,0.938356,0.729167,0.875,0.979508,0.795455,0.854337,0.759517


## KNN ##

In [139]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler


# Define the hyperparameter grid to be searched by the grid search
knn_param_grid = {
    "kneighborsclassifier__n_neighbors": range(3, 30, 2),
    "kneighborsclassifier__weights": ["uniform", "distance"],
    "kneighborsclassifier__p": [1, 2]
}

# Run a grid search to find the optimal combination of hyperparameters
knn_search = run_grid_search(
    KNeighborsClassifier(),
    knn_param_grid,
    X_train,
    y_train,
    preprocessor,
    StandardScaler(),
)

# Display the grid search results
knn_search

Fitting 5 folds for each of 56 candidates, totalling 280 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'kneighborsclassifier__n_neighbors': range(3, 30, 2), 'kneighborsclassifier__p': [1, 2], 'kneighborsclassifier__weights': ['uniform', 'distance']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time f

In [140]:
# Evaluate the K-Nearest Neighbours (KNN) model using the testing dataset and obtain performance metrics
knn_scores = evaluate_model(knn_search, X_test, y_test)

# Label the metrics to indicate they belong to the KNN model
knn_scores["Model"] = "KNN"
knn_row = pd.DataFrame([knn_scores])

# Append the KNN metrics as a new row to the existing DataFrame of model scores
model_scores_df = pd.concat(
    [model_scores_df[model_scores_df["Model"] != "KNN"], knn_row],
    ignore_index=True
)

# Display the updated DataFrame containing all model performance metrics
model_scores_df

,Model,Accuracy,Recall,Precision,Specificity,F1 Score,Balanced Accuracy,Cohen's Kappa
0,Decision Tree,0.938356,0.729167,0.875,0.979508,0.795455,0.854337,0.759517
1,KNN,0.941781,0.6875,0.942857,0.991803,0.795181,0.839652,0.762215


## Random Forest ##

In [141]:
from sklearn.ensemble import RandomForestClassifier

# Define the hyperparameter grid to be searched by the grid search
rf_param_grid = {
    "randomforestclassifier__n_estimators": [100, 200, 300],
    "randomforestclassifier__max_depth": [4, 6, 8, 10],
    "randomforestclassifier__min_samples_leaf": [3, 5, 10],
    "randomforestclassifier__max_features": ["sqrt", 0.5]
}


# Run a grid search to find the optimal combination of hyperparameters for the Random Forest model
rf_search = run_grid_search(
    model=RandomForestClassifier(random_state=42),
    param_grid=rf_param_grid,
    X_train=X_train,
    y_train=y_train,
    preprocessor=preprocessor
)

# Display the grid search results for the Random Forest model
rf_search

Fitting 5 folds for each of 72 candidates, totalling 360 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'randomforestclassifier__max_depth': [4, 6, ...], 'randomforestclassifier__max_features': ['sqrt', 0.5], 'randomforestclassifier__min_samples_leaf': [3, 5, ...], 'randomforestclassifier__n_estimators': [100, 200, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls th

In [142]:
rf_search.best_params_

{'randomforestclassifier__max_depth': 10,
 'randomforestclassifier__max_features': 0.5,
 'randomforestclassifier__min_samples_leaf': 3,
 'randomforestclassifier__n_estimators': 100}

In [143]:
rf_scores = evaluate_model(rf_search, X_test, y_test)
rf_scores["Model"] = "Random Forest"

rf_row = pd.DataFrame([rf_scores])

model_scores_df = pd.concat(
    [model_scores_df[model_scores_df["Model"] != "Random Forest"], rf_row],
    ignore_index=True
)

model_scores_df

,Model,Accuracy,Recall,Precision,Specificity,F1 Score,Balanced Accuracy,Cohen's Kappa
0,Decision Tree,0.938356,0.729167,0.875,0.979508,0.795455,0.854337,0.759517
1,KNN,0.941781,0.6875,0.942857,0.991803,0.795181,0.839652,0.762215
2,Random Forest,0.955479,0.75,0.972973,0.995902,0.847059,0.872951,0.821516


Random Forest achieved the best F1 score and balanced accuracy.

Decision Tree had strong recall but lower overall performance.

KNN performed well but did not outperform Random Forest.

# Test models for overfitting #

In [144]:
def compare_train_test(model, model_name, X_train, y_train, X_test, y_test):
    train_scores = evaluate_model(model, X_train, y_train)
    test_scores = evaluate_model(model, X_test, y_test)

    train_scores["Model"] = model_name
    train_scores["Dataset"] = "Train"

    test_scores["Model"] = model_name
    test_scores["Dataset"] = "Test"

    return pd.DataFrame([train_scores, test_scores])


In [145]:
overfitting_check = pd.concat(
    [
        compare_train_test(dt_search, "Decision Tree", X_train, y_train, X_test, y_test),
        compare_train_test(knn_search, "KNN", X_train, y_train, X_test, y_test),
        compare_train_test(rf_search, "Random Forest", X_train, y_train, X_test, y_test),
    ],
    ignore_index=True
)

overfitting_check[
    ["Model", "Dataset", "Accuracy", "Recall", "Precision", "F1 Score", "Balanced Accuracy"]
]

,Model,Dataset,Accuracy,Recall,Precision,F1 Score,Balanced Accuracy
0,Decision Tree,Train,0.974315,0.875740,0.942675,0.907975,0.933365
1,Decision Tree,Test,0.938356,0.729167,0.875000,0.795455,0.854337
2,KNN,Train,0.964897,0.810651,0.938356,0.869841,0.900821
3,KNN,Test,0.941781,0.687500,0.942857,0.795181,0.839652
4,Random Forest,Train,0.988870,0.946746,0.975610,0.960961,0.971371
5,Random Forest,Test,0.955479,0.750000,0.972973,0.847059,0.872951


The Random Forest still shows some overfitting, especially in recall, but it now has the strongest test performance of the models tested. The overfitting is acceptable for this stage, though I would still monitor recall because the model misses more expensive houses on test data than on train data.

# Competition test dataset #

Based on test F1 score, balanced accuracy, and overall accuracy, I selected Random Forest as the final model.

In [ ]:
comp_data = pd.read_csv("data/1_classification_competition_test_set.csv")

In [147]:
comp_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1459 entries, 0 to 1458
Data columns (total 80 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   LotArea        1459 non-null   int64  
 1   LotFrontage    1232 non-null   float64
 2   TotalBsmtSF    1458 non-null   float64
 3   BedroomAbvGr   1459 non-null   int64  
 4   Fireplaces     1459 non-null   int64  
 5   PoolArea       1459 non-null   int64  
 6   GarageCars     1458 non-null   float64
 7   WoodDeckSF     1459 non-null   int64  
 8   ScreenPorch    1459 non-null   int64  
 9   MSZoning       1455 non-null   str    
 10  Condition1     1459 non-null   str    
 11  Heating        1459 non-null   str    
 12  Street         1459 non-null   str    
 13  CentralAir     1459 non-null   str    
 14  Foundation     1459 non-null   str    
 15  ExterQual      1459 non-null   str    
 16  ExterCond      1459 non-null   str    
 17  BsmtQual       1415 non-null   str    
 18  BsmtCond       1414

In [148]:
comp_data.drop(columns=["Alley", "PoolQC", "MiscFeature"], inplace=True)

In [149]:
comp_data = comp_data.set_index("Id")

In [150]:
# Use the best estimator from the Random Forest grid search to make predictions on the competition test dataset
comp_pred = rf_search.best_estimator_.predict(comp_data)
pd.Series(comp_pred).value_counts()

0    1264
1     195
Name: count, dtype: int64

In [151]:
# Create a copy of the competition data and add the predictions as a new column
comp_data_pred = comp_data.copy()
comp_data_pred["Expensive"] = comp_pred

comp_data_pred.head(15)


,LotArea,LotFrontage,TotalBsmtSF,BedroomAbvGr,Fireplaces,PoolArea,GarageCars,WoodDeckSF,ScreenPorch,MSZoning,...,Functional,GarageType,GarageFinish,GarageQual,GarageCond,PavedDrive,Fence,SaleType,SaleCondition,Expensive
Id,,,,,,,,,,,,,,,,,,,,,
1461,11622,80.0,882.0,2,0,0,1.0,140,120,RH,...,Typ,Attchd,Unf,TA,TA,Y,MnPrv,WD,Normal,0
1462,14267,81.0,1329.0,3,0,0,1.0,393,0,RL,...,Typ,Attchd,Unf,TA,TA,Y,NaN,WD,Normal,0
1463,13830,74.0,928.0,3,1,0,2.0,212,0,RL,...,Typ,Attchd,Fin,TA,TA,Y,MnPrv,WD,Normal,0
1464,9978,78.0,926.0,3,1,0,2.0,360,0,RL,...,Typ,Attchd,Fin,TA,TA,Y,NaN,WD,Normal,0
1465,5005,43.0,1280.0,2,0,0,2.0,0,144,RL,...,Typ,Attchd,RFn,TA,TA,Y,NaN,WD,Normal,0
1466,10000,75.0,763.0,3,1,0,2.0,157,0,RL,...,Typ,Attchd,Fin,TA,TA,Y,NaN,WD,Normal,0
1467,7980,NaN,1168.0,3,0,0,2.0,483,0,RL,...,Typ,Attchd,Fin,TA,TA,Y,GdPrv,WD,Normal,0
1468,8402,63.0,789.0,3,1,0,2.0,0,0,RL,...,Typ,Attchd,Fin,TA,TA,Y,NaN,WD,Normal,0
1469,10176,85.0,1300.0,2,1,0,2.0,192,0,RL,...,Typ,Attchd,Unf,TA,TA,Y,NaN,WD,Normal,0


In [157]:
submission = pd.Series(
    data=comp_pred,
    index=comp_data.index,
    name="Expensive"
)

submission.index.name = "Id"

submission

Id
1461    0
1462    0
1463    0
1464    0
1465    0
       ..
2915    0
2916    0
2917    0
2918    0
2919    0
Name: Expensive, Length: 1459, dtype: int64

In [158]:
submission.to_csv("submission.csv", index=True, header=True)